# 🚀 Milestone 1 — Crawling Benchmark & Metrics Evaluation trên Colab CPU

> **Mục tiêu**: Thu thập toàn trình bài báo arXiv (**Discovery → Metadata → Sources → References**) cho từng thành viên nhóm (`OWNER`), đo đạc đầy đủ 100% các metrics theo yêu cầu chuẩn Rubric của đề bài (`context/crawling_new.md`), áp dụng cơ chế tăng tốc **Semantic Scholar POST Batch** (nhanh hơn **~57 lần**, loại bỏ hoàn toàn nguy cơ nghẽn HTTP 429), tích hợp cơ chế **Checkpoint & Resume siêu tốc** qua Google Drive (đáp ứng trọn vẹn quy định chia 2 phiên chạy tại Q&A Hàng 15).

---
### 📋 Các nhóm chỉ số bắt buộc theo Rubric Đề bài:
1. **Running Time (Thời gian thực thi)**:
   - Tổng thời gian toàn trình (`wall_seconds`).
   - Thời gian truy vấn danh mục (`discovery_seconds`).
   - Thời gian trung bình xử lý mỗi bài báo (`mean_paper_seconds`).
   - Độ trễ nhỏ nhất / lớn nhất của từng bài (`min_paper_seconds`, `max_paper_seconds`).
2. **Memory Footprint (Mức tiêu thụ bộ nhớ & lưu trữ)**:
   - RAM cực đại (`peak_ram_bytes` / Maximum RSS đo qua `psutil`).
   - RAM trung bình (`mean_ram_bytes` / Mean RSS qua lấy mẫu 50ms).
   - Ổ cứng cực đại (`peak_disk_bytes` / Peak Disk tại các mốc tải/giải nén source).
   - Dung lượng lưu trữ đầu ra cuối cùng (`final_output_bytes`).
3. **Data & Crawling Statistics (Chất lượng dữ liệu thu thập)**:
   - Tỷ lệ thu thập bài hoàn tất (`paper_success_rate`, `papers_complete`).
   - Độ bao phủ phiên bản (`versions_expected`, `versions_covered`, `unsupported`).
   - Dung lượng trung bình file TeX trước và sau lọc ảnh (`mean_before_filter_bytes`, `mean_retained_bytes`) & tỷ lệ giảm dung lượng.
   - Số lượng trích dẫn trung bình mỗi bài (`mean_all_references_per_fully_fetched_paper`, `mean_arxiv_references_per_fully_fetched_paper`).
   - Tỷ lệ chuẩn hóa Metadata trích dẫn (`metadata_macro_success_rate`, `metadata_micro_success_rate`).
4. **Platform & Network Audit**:
   - Thông số máy Colab CPU (CPU Model, Cores, Total RAM, OS).
   - Nhật ký HTTP request distribution (`http_counts`) & tỷ lệ 200 OK.

## 1. Cấu hình Tham số Thực thi

* `OWNER`: Chọn mã thành viên tương ứng (`'trong'`, `'khang'`, `'hoang'`, `'man-nhi'`, `'huynh-nhi'`). Mỗi thành viên chỉ cần đổi đúng tên mình.
* `COUNT`: Số lượng bài báo cần benchmark:
  - `10`: Test nhanh môi trường & kết nối API (~1-2 phút).
  - `100`: Chạy mẫu thử nghiệm chuẩn A/B báo cáo (~15-20 phút).
  - `5000`: Chạy toàn bộ dải Milestone 1 chính thức được phân công.
* `VARIANT`: Mặc định `'batch'` (S2 POST Batch - siêu tốc gấp 57 lần, chống lỗi 429) hoặc `'baseline'` (GET tuần tự).
* `SAVE_TO_DRIVE`: Tự động mount và lưu trữ kết quả (.zip, metrics, biểu đồ, checkpoint) lên Google Drive của nhóm.

In [ ]:
import os, sys
from pathlib import Path

# ==============================================================
# CẤU HÌNH BENCHMARK CHO TỪNG THÀNH VIÊN
# ==============================================================
# Chọn mã thành viên của bạn:
# 'trong'     : 2504.17523 -> 2505.00665 (5,000 bài)
# 'khang'     : dải phân công của Khang
# 'hoang'     : dải phân công của Hoàng
# 'man-nhi'   : dải phân công của Mẫn Nhi
# 'huynh-nhi' : dải phân công của Huỳnh Nhi
OWNER = 'trong'

# Số lượng bài cần thu thập (10 / 100 / 5000)
COUNT = 5000

# Chế độ thu thập trích dẫn:
# 'batch'    : S2 POST Batch siêu tốc (100 bài/request, nhanh hơn 57x, chống 429) -> KHUYẾN NGHỊ
# 'baseline' : S2 GET tuần tự từng bài
VARIANT = 'batch'
BATCH_SIZE = 100       # Kích thước batch cho Semantic Scholar (1..100)

RUN_ID = f"m1-bench-{OWNER}-{VARIANT}-{COUNT}"
WORK_ROOT = Path(f"/content/m1-work-{OWNER}")
CORPUS_DIR = WORK_ROOT / 'corpus'
SAVE_TO_DRIVE = True
DRIVE_MOUNT = Path("/content/drive")
DRIVE_OUTPUT_DIR = DRIVE_MOUNT / f"MyDrive/arXiv-data/milestone 1/benchmarks/{OWNER}/{RUN_ID}"
DRIVE_CHECKPOINT_DIR = DRIVE_MOUNT / f"MyDrive/arXiv-data/milestone 1/benchmarks/{OWNER}"

print(f"==> Cấu hình đã chọn:")
print(f"  - Thành viên (OWNER): {OWNER}")
print(f"  - Số lượng bài (COUNT): {COUNT}")
print(f"  - Cơ chế (VARIANT): {VARIANT} (Batch Size: {BATCH_SIZE})")
print(f"  - Thư mục làm việc: {WORK_ROOT}")
print(f"  - Thư mục corpus: {CORPUS_DIR}")
print(f"  - Lưu Drive: {SAVE_TO_DRIVE} -> {DRIVE_OUTPUT_DIR}")


## 2. Kiểm tra Môi trường Colab CPU & Nạp S2 API Key

* **Xác thực CPU-only**: Đề bài quy định benchmarking thực hiện trên môi trường CPU.
* **Nạp S2_API_KEY**: Lấy tự động từ **Colab Secrets** (biểu tượng chìa khóa ở thanh công cụ Colab) hoặc biến môi trường/nhập trực tiếp, kèm kiểm tra kết nối (preflight check).

In [ ]:
import os, sys, platform, subprocess, json
import psutil
from urllib.request import Request, urlopen

print("=== 1. KIỂM TRA PHẦN CỨNG BẮT BUỘC (COLAB CPU) ===")
gpu_detected = False
try:
    res = subprocess.run(['nvidia-smi'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if res.returncode == 0:
        gpu_detected = True
except Exception:
    pass

if gpu_detected:
    print("⚠️ CẢNH BÁO: Đang bật GPU! Rubric đề bài yêu cầu chạy trên CPU-only instance.")
    print("   Vui lòng vào: Runtime -> Change runtime type -> Chọn CPU (None).")
else:
    print("✓ Môi trường hợp chuẩn: CPU-only (None) - Đạt chuẩn Rubric đề bài!")

total_ram_gb = psutil.virtual_memory().total / (1024**3)
cpu_cores = os.cpu_count()
print(f"✓ CPU Cores: {cpu_cores}")
print(f"✓ RAM Tổng: {total_ram_gb:.2f} GB")
print(f"✓ OS / Kernel: {platform.platform()}")
print(f"✓ Python: {sys.version.split()[0]}")

print("\n=== 2. NẠP & KIỂM TRA SEMANTIC SCHOLAR API KEY ===")
s2_key = None
try:
    from google.colab import userdata
    s2_key = userdata.get('S2_API_KEY')
except Exception:
    pass

if not s2_key:
    s2_key = os.environ.get('S2_API_KEY')

if not s2_key and Path('.env').exists():
    for line in Path('.env').read_text().splitlines():
        if line.startswith('S2_API_KEY='):
            s2_key = line.split('=', 1)[1].strip().strip('"').strip("'")
            break

if not s2_key:
    try:
        from getpass import getpass
        inp = getpass("Nhập S2_API_KEY (hoặc nhấn Enter nếu đã lưu trong Secrets): ").strip()
        if inp:
            s2_key = inp
    except Exception:
        pass

if s2_key:
    os.environ['S2_API_KEY'] = s2_key
    try:
        headers = {'x-api-key': s2_key, 'User-Agent': 'arxiv-data-science/T4 (academic course project)'}
        req = Request('https://api.semanticscholar.org/graph/v1/paper/ARXIV:2106.15928?fields=paperId', headers=headers)
        with urlopen(req, timeout=10) as resp:
            if resp.status == 200:
                print("✓ S2_API_KEY: Hợp lệ và sẵn sàng hoạt động (HTTP 200 OK)!")
            else:
                print(f"⚠️ S2 API phản hồi HTTP {resp.status}")
        import time
        time.sleep(2)
    except Exception as e:
        print(f"⚠️ Kiểm tra kết nối S2: {e}")
else:
    print("⚠️ Không tìm thấy S2_API_KEY. Vui lòng thêm vào Colab Secrets (🔑) để đảm bảo không bị rate limit.")


## 3. Thiết lập Mã nguồn Pipeline & Adapter Tự hành

* Tự động kiểm tra mã nguồn repository, nếu chưa có sẽ clone từ GitHub `arxiv-data-science.git`.
* Cài đặt các thư viện cần thiết (`psutil`, `matplotlib`, `pandas`, `tabulate`).
* Tự động kiểm tra và đồng bộ gói adapter S2 Batch và Resume vào thư mục `tools/`.

In [ ]:
import io, base64, zipfile, subprocess, sys, os
from pathlib import Path

REPO_URL = 'https://github.com/ChickenDev-cyber/arxiv-data-science.git'
current_dir = Path.cwd()

if (current_dir / 'src/arxiv_pipeline').exists():
    CODE_ROOT = current_dir
elif Path('/content/arxiv-repo/src/arxiv_pipeline').exists():
    CODE_ROOT = Path('/content/arxiv-repo')
else:
    print("Đang clone repository từ GitHub...")
    subprocess.run(['git', 'clone', REPO_URL, '/content/arxiv-repo'], check=True)
    CODE_ROOT = Path('/content/arxiv-repo')

print(f"✓ Thư mục mã nguồn: {CODE_ROOT}")

# Cài đặt dependencies
req_file = CODE_ROOT / 'requirements.txt'
if not req_file.exists():
    req_file = CODE_ROOT / 'src/requirements.txt'
if req_file.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req_file)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'psutil', 'matplotlib', 'pandas', 'tabulate'], check=True)

# Tự động bổ sung các module batch adapter nếu chưa có
tools_dir = CODE_ROOT / 'tools'
tools_dir.mkdir(exist_ok=True)
BUNDLE_B64 = '''UEsDBBQAAAAIABOzdl1O7iKX+QwAABYmAAATAAAAbTFfYmF0Y2hfc3VwcG9ydC5weZ1a63LcthX+r6dA7B8kZYrald3OdBW6VWy3USeJNZKSSavZ4UAkVouKS7IAKGnj0bv0XfpiPQcXEuRyHU35Q17icnBwLt+50K9evfrcqCNekR/nhD01TPANqxShBW0UE/KUqDUjeb3ZcKVYQRresJJXjDQ0v6d3jHBJ2ipf0+qOFcmrV68OVqLewIZKsSdV8lvCN00tFKnasrSjbkmzdZMFYw2+m5mCKqaAjW7Wvh/Y97VSTZKXHPh0Q/+SdeV+19KQaahaewxcwKtbotaC0YJXd90Akte7WlHCpqShQnYMwBir8rpgMfySTcnVgVlMxRN/yJxMkoLLvH5gYpvQhrvNZw2/ZLKpK8niS7apFfskRC0mCQi2YgKOYtKncMU2tFI8v8rXdUnFB33z+Ook++v5px8+XsXZBc1ZYYZ/j6xk4oHn3c0uu5kf66ItWZy1Ks+q+nGSDi/gBL7iYBedWmuxoSX/jWVmLS9iLrMHGCq6kWme2qpiYszPhZ2+1LNf27gBdZQdGw0Fy81kw/KDg+/Orj98n/18+UMaoKXIxfExCBNOMlKURopJLe6O7wRt1scP82O9//iWqnwdWAJGuGmgp86LGAyXiYqW54WMO4F+qFtQhSdfxRVI0ddjq9a1kP7QllHhvzftbclzqnhdfQRT96ceWNUOBjwugoMfz37Nvvv88R/p/I+H89nJO/3n4OCgYCvS1FJlStBKooBCMNwYjbxuVbwG2wcVxrd1sY0WBwQeRcUdU6kzb1we6Qm+IqGZTEBwbMO+sVINSC3stmQNZ1VUz+2RdKCJ4dPvsvajCIBP+FNdsfjdu7eRt6CVeFXAAW8PlfKxFoVlGx9BOXjqL7RsjWOFwdUJyQXTtkrBRDatVEQAT3BOXWk8+/76+uKKnF2cA2l+x6vAXBbwqWI56iH1MCbRqz90c+Ho1k6uqf3X0FJi2/PYEwZf/HfLpAqDi89X10Hc3Uut3wR/Dt7Yd1gjtlpBKf5xKkvtv1FH+ZGrtU8eNguLNmFEqCTurWcGn4Zuy5oWqZtNEA9DZ09v5hEqvpuUiqpWpunJbEbA6Ri5DYIBOVgMCBlaqtF7R2hhlOPBXhjwyqCDXRzEwXfodkQCiJCSQ5yBOJQz0B/MXYuWRYOjBFOtqHxUDUd8xgXPVT8IErFSC6ModixqmnhOo8jnK81aL6HX5CcGMA6GmZdtwQglVmvHhhIp2k2DZmsIcLQqcMykI+BHEoa0HU4BKhn0H1qvG9a2i4bhqZSsKC9BFpGhio6id684wEA5bWR5WaP+AQryEjyGaAlfgL2aIBH6EcO6EoTun4GTRrAcOCq32k/YE5cKwiQBtiDgw68YFSC2hFYFkWsKbkYQFzB2CLKCiyL3Og9AoghEWcYrrrIsBKqr+DAGhMju2TbVHt8BlHkFSKfbdB4fHt4/giPIyNfJJQNAGLKlLYliXgK+fARUnYJAFlKJ1kijo4FYnE5G09BxZf/tbc553pfDQ9yeOPAMPmAyU6mj623DggUAX+NA/BizkeC5I4EXTzIE5LS7L2LaEKJHy43ve35mh1GTWVnn92mXxCQ/wGvYs4xSl6wqBqAPAypz8LGYcqmezb0b4563/jjZQoAMo6RTtNGi/jsGLk/hyGE84GPw3Im6bdLAxZLMBpOgt42DzsRWDIRi7IsX/u0Al3SAKSTKGyEK59/PAcfgHafAhcKdhCWkkbZlij4+pNj7rB9xDIIhSHDAnXmS4Ak5reoKTKIk5x9lMILssTaHJ3gmgLaUIN7I8EsAvASLm+Ds8tfzXxbBGzrgcvkcJSZNDUeQSR9TQxGwMOzSIx1vutQWqENiV+IBfv7zHPW0MKS5wKHZwl8yBPL9GgupYS+d+GcQAhzwkeFfPRS9LC6cmxGipUT+fvX5px0Q9JUsOfg8hSzJxaG4BJiInOYd8n+TOjuY0uqLQpWLLaBiUSAMc7UlGy43On/shWEdy27v0PgC8zmFqHmpvWKxBytXYJy3UGZ5jGo1uvHU/RhOIyps0y8j/NFhSO4M36LxsSLtJPqXRtTgz2rbMUWh8Ns0SmqmQHUeWjgGErem29SnrCC9O2YuRKE6qFcryVSsA/3QU81Mms50cIG1aNf9jdzBXwJAfBosXM0Y9ksgpWxC2BhFzz5h/6JYr6K14H2tFfrTL+PHyNLfbkZuYNFyrP2hoMZy2RHJMGaP67NwPBDtMx6hTSvG6AiZvsYZk85OBVfNY2PNMh3bZ2hoRSPL8Wim/sv+wDA4xrI2AeeZW2Juoqi8nwB1PWyNop8rZHqDMzeBA/NgqVESB1GBettyxzzckU63A2h4Tc5uMWZjUrglbfUAWINJh+RQuEEaQldQlIGVmK4JfwBgWPOyIGuu3p38KRnQ0vTTCtLFEGiIcOr05AHRE/PVIZBrtDd2CBddTG3VJpiyQY7ZG+MIyhuMzDvKTIzsESHHsO6hI9YWMDoUFFaLiK5GZe/n2nNCWJbwalVjKh7q6jEzmTrAOdZ+72az+N387fCu7oGTgr40CGzW7dHbMCnBkwCcgwgi0iOaXDTCdnw2vChKlvbsHR+f7CyyOZCzP73wZmG2LqPTyWkzu4DpkZCtRL5y/TQF+9hldahYB9FAZ2fla3LVihUk8zovhiwdi4VW4CvkT3dr258hpmkC+pNtqWRi6psJardbrLBhN8BkXtdlUT9WJmGh5G+frlEbEOVKTPZ1maLthslkh9TLTXV0q9fkM1xFGD+RpEv5QWTgbHKNPUgmjrTxapawdE6+ZuvO+2OzBzj6jTdGd2ZonAlMJRQNYinUldECm5i8atnQq21bJtUrjab9Vk30Owe4teYMk5yOG2vdIkP+TPzKH4II8pkx3u0a1CTPOfaufIaHXa0A7XklJxdMXEhBGRRqkpGLsbzSZY4e/LbLuL1bI/1hkoYjcCO95YXXgLNptQ13SVtZovrhDfWuqU8r8FFAAQdOdvMlyBHbL/DWXpIBW6PnMa3lmBFj5oBGGXhFft/UfOoaU6sSSR9YaDQJPgdnL+OxXuN+IyKieTVJ+3QhBZcxuUWwmMUBxpzMvetyO4DUPYPbbiDptwKIAxv6MwqrXDc4jJ6XQ41jG2jk1ToF2wm+y9RSHpVrmZcJaRTxgn1sPxTEhzFko3eQbkudoPoZi6IC1JT2HJ7eMlAPm4hng+QUH1UXdXqz3EGILj8YVWOgmiKd1Brm9f+/1qJ9NmO6bIMqB5/XWDpgxxw/vwjsbhLcCNkGBcM/7fwDEXMzDgGAlcmOveLN0F11OoxiSVBXlbnTKOHz4h4s7Cd1G8PJ+CviPzIK8lOvwdb3NtEGZ1aD9M32iCeszWiuU1s0Tts+QdgVEBybGko0SNVND8vma30AI4YzEIRscd5l6ygzOSCJTZFEloAK4dvkD8PSeBSDdKRNXQ68x+aH5t4Zu/uxg7KW7ML+ezNbgqSs/ILlm3Qg0J3c0ewy6GXo7MCGmbgJrH8hCixT+7J3cdEK3fjKJNT4FYS7ZbqhT+EsiUP37S7Bup3LeoXGqLuzeucKKgNp0WYJOm1KyGXC4J+Qzb2ZzRazGQS4o/1Noq88ew62V9lzVJSoWtHS3SOMdnNSxzmo6IFVGGsC0EH/5TSTJ7qHZhQMkgiUaFnwQjqmpZoN1Ai4gB8FikzV2YoLmDF7tWJEOFj7Un5BMmDoGfpFvkUmbS/XuEMJsqvy7Skpau18st2YL2zEaVpi+v8IbhKMrF6XvNa6enwdt6rxmXDnJC8ZBW85nSyKpicHDYxhv3vwLTEcvu6tmw+xGLUFM+J1OofqZKJkdilPv3Kc9+DP+bdpv+BbpLXY7R72K0znMJistDUz/c/9Vba5wVRdPWg+SNfQ8RoW+I0M7MKAuHYFEDumP9Fp0zUFdnpIYOpwi74zEcSzGJM5G9l0hh23wFwa3P73P9y7HoISrwr2FLvwy6p2w8DGmL87no+QHTv1vxN2wU4alqf9h2G9Pv5qDhsCXRdt+vuabx2myQQ4n5lg4VzORc9JkOp22k6viQLgC1CmFvh9rpv3VQPxF5QQItOThSx94pt2k45sY2edyy2kqptMh7xMh5tBC8wRAzgI7e94cteR6XM9sbwF/NzFGdOvv1nuTKCOc7io/lDTx3Gt2But/KP5YqJmsVdAQ9K0o/epZXBxC2hxv2/DXnl6ptDxM7SHaKFPcjlQt2r3tnCOXjrNtmNBf3lfBZ+64ABpWJ91LMiX/nLPBmDHRukLsY/emv5G9/+k/5879iQZ+gSj1Z1Mw2d1wtbc03vDTRe7nee5d8/3lqkZHNBDCOryZV20xujH+LVt92A4sMGIlPYn6+4urI51KTAR6Eyz3W7U6WX3u7d4k1q/VIIGNHzRjb9G4zWtFEF/mGVox437r8xDq9jBSoN/L8PLnoyukDO92PDY82xlP7VLqyxzSdcIQTv4jw7+B1BLAwQUAAAACAATs3ZdxSEFtcwFAADSEgAAEwAAAG0xX2JhdGNoX21ldHJpY3MucHmdWEtv4zYQvudXELlIQrTZ7VUL9dZrD0UPBQJDoGU6IVavJSmv3SD/vTPDpyQ7m60PjsQZzuObp3N/f//HeepkKw3jWsvnoReDYb0wSraa/ZDmhU3zHhi4kePwqRMn0TEt/xXsIIaxlwM3o9KP9/f3d0c19qwdu060yKuZ7KdRGWA88rkzB9mau7s7eGP8JBR/FvmJd7PQRaWEmdXA9Nz7o8+dGPwzk0dmH5notGB/joOwchQalfNyH0Twz3tk36ecxAqye67A7ia6mRuuv+lSCQ3mgRl3DD4dN0Kb+vWN3o6jYorJgTkmyxP5nnL1+CxMno0/BqGyonSvXJ3lqZGHeKKNmOIbAKABo6wodrUimZNQzcThu37afRXnCUAUh7odgRP+zoOeJ0STnsGYsTvB4xe6uhdgpmgwKhpv86MBYeH1OHddo/iP8EzG4VsvWzU2z+MIkr72HF7glES2Yz91wghrknaKEA3EDAEh7CIc5H/J5aFGwpPDY1fatwDHLlyAFOO1BZEgyYOEMkPagRuelRjBonx9K8I9PK+RwQJJbAUDw1zE8OPQ1TVSF4DrrHzaRWF6nFWLKN0wxNGz8vQUQrYjewiLEwLhJUfXfPQeaspiR49aXVCbF/iuMem1TxFuZg3egFR4m1tQDbqzJPiZVayRxRm3kftQpwoCNZHycE1rXS8UvaMnJuDKw09XFftEDAckmYQm4mMqUXbU9tia6NrRJs5UN1yqOu8tozgbxRH6Zn+BgDauMo6yM1Sdjg2aBZcDcFkllhlqcSEY2gjvutxcJpGfbUzkYMpjN3JTMD4c2Pn3+gs5c0YqGlJU5OwjnyYxHHI6WuDQjN/qfD+OXQSNRG0TtfEliPmg2d9qFgvz6INXF9m0jAfR8YSsKlbUaJmFySbF05ddEnq6h1A445O+GmzAbuPu/vaLdyOxSpuYx8+eFWlD8yQ6ih4ocdS3ugnQAJqB6njbT9ShxsvvdBMkh2jU6mBZ3wkPpA3wz4Opbd+zz6Gl2tc1Dgsty2LA3u3VwnOTOLRMWZhpSkKtBeZbjKBPajlA6QMxB5llJ7XL6oTg5JU4uotqk37RT8wrkAKRShzGQyeh2NxdoYHpg9WG/Bcy/hvhOoyGMobSCufOtgbgk2dGmk5Aq+SzeYF9BJ70vO+lxkxvIKpI06Lng5Fto9uXseOKxrPtp6QWk9bZ+2j3jbzYGu6naUhED0KRDNdAjHAUybR9WCXDRgn2nni1otHsZS6vfk5VBDk00vM4JZMe78fK/247eO9KB/tZG4pFdmvMiX4ylyxyB1PSCbdYSx5qdDRQwwrloXp1a0hlu0FcQypqDMFnP7CzamF3ufA8hYhczarU5zJbDBs3U1xbK+PIcefUv1YKYrnGKFQLJMp1/Vch/bx3C2LMjZWqAGRgb8LSVS0zrMwI08Qkij4iX7g+6pbv1yz5qaAtmLQmFk5G6tcqlNE+p87lBZQaoUCb/vpOVJDEKq561WsWI+ufymwdvuWCVSUvSPGrDhL8c3m1C3kfqe0Mz42zo3mB1jqqS1Zhk8OGlqeDyiS7crooX92Ct1OquF2+tK2HVb14SyByew9NVIRpHqTJKviD18FLLaDGMVG/4g8nbPRhzcUqS38SZtfBsGo4GtNssyId9pAc4tx28wGK41b+fNpeuq21F3xorhaj/925Uk8XNgXqeJPdYwFhWmevGbb+S3MUpn1BNygRrAN+XhSYTJ5hxEaljOTdgjdxNl674Wj2fRbqcrVMlvevFslKlhkNJ/bmSg9qcDfQNn1/bhaBiV0pkYT1fAWhCPJHxa463AcF07X3RNs6a2jMrhD1gohWlBtml+FJEB3nz7XJrTYbv7golAFz68MHxIaoWbDcQpNVidDIvOBpkv/oJNH+KHrTCKULXS77S3yfJfRVWGH+kScWwsWOUnQHaCmEEHNFrxnNfceshUEGtJVNMGa038iytPZo0CctqAqT/+3uP1BLAwQUAAAACAATs3ZdFs6mNZoCAAAnBwAAGgAAAG0xX2V4cGVyaW1lbnRfc2NoZWR1bGVyLnB5pVVRb5swEH7nV9yyB6CjiLbTpHXLQ9VF2qStm1ptmhRVyIMjsQI2s00a/v1swARC0k2aXzD47rvvvjsfs9nsa6nOKQMiftItvL58C79IsuFZ9g7UGvWLxJwyhJKW7UYma0yrHAVIRWoJFUvWhK0wDWezmZMJXmisHd3G1iNcK1UCLUouFNzj7wqlerAgATzcflx8+P55cR/f3XxZPHy7uV04jpPkREpY7EoUtECmDv28ww/+tQN6pZhBHFNGVRx7mnoWwNnZ5omIlewszDIHYdzSNCnPIdqfVTqm54c9Su/v7CNIRkq55qoLsRK8Kgf4JalzTlKN26P1Hq1tb0qz1hvmc3AbRu4eZ4C1dBPOJCaVols0pN1HAz9OpHcUqCrBrO+AuNCacYEN76EglqY99wdQCRepjaXlTjYlp0zJ0EB7R8oXdLQ2WHtdSv4o4Q6SSmBcwR1nOE5ZKoFko0O2hku3y8N9DFeovIkSAUT+CEDHUHWps2yAfBtJswYumm0E7+c2jt5dRFE05tCkTqhE+EHyChdCcOG5n9iW5DQd3JYOQ98g7JsV9iq5Y2LTxmv99xUSRGGc04IqTIfdFZiaijommeqbfdQ/L070T9cKtsIj/CPIve8TVeuOb86TjVw2xo/Xf8mnoMw7/PoKLoJG4rEYGclzM2w6r6soCgN4E4VwBpf61h6FOjdQV/4YKMWc1AaF7DwLeiIrs14aofX80gaEMtnMuRRJ2ow3XcgCCy5qwC0yI6/WTVKpKFtpxjSX4VSBBs+q2bDxj9WgORmUuh1hsaySBDEd1/t/SjzBPZg5/1RaHfVA/OkFKQVuKa/kM5PodKdEExtdr2mMvXNXiMNs7MJdgqXS/wzzoJw9BzXkYZM4at6MAOcPUEsDBBQAAAAIABOzdl0HGF4OaA8AAIwxAAAWAAAAbTFfYmF0Y2hfZXhwZXJpbWVudC5wea1aTXLcuBXe6xQsZwFSQ1GSK0kl7cFUPDOeTBYzdlnepBQVCyLRao7YJEOAsnscHyJHyCKLVG6QLLLISeYmee/hhyD7R21PtFCTIN4D8OH9A0+ePHnZ6bOqiZZDXZ99dxndCiXrqpHnt0IXq+hWNsVqLfr7LPqml2oV9W2r1bOoaaNiJZo7qSLdRmtRNdnJyYt3Yt3VchG9+uObb19+/+r5m2+56ouo2+hV20DHtlbn68ucWOfyXSf7ai0bnXWb6OzsQfSVaHREX0+gAYeKzou20dAH6M7oy/ktfhqas6o0Xc8uLy6gqX3byD7SfdvcwVvRDo2GDydPnjw5WfbtOiraupaFrtpGRdW6a3sdfYWdZG++l0JLDdNxH917iv9+bBtpunVCr+rq1vV6Ba8n9ln0d53olUxXQmGf9AfVNmmrUjXcdn1bSAWPG0UMDbOhr6Ff1ss/D1JpxxNa2042ky6y79vedfj2zZtXL7DhxPQR/bvqIe+qjnYuG/cMRl1WtfSEd1LnZdXnqvpRPkIKCDcj4Zeu/Ur2D1UhU/lO96LQOXTLYT1Vc6d2MjRssnVbytrj/npoQOpwJ9Lv2nKo5ZXUh6iVGdRjbj+/pq+HCIu6ckTFShb3XVs1On8QdQW72+4mVVroSumqUPNxr/wXi4Kh9wKtho76Ocyw8VWr9Fd1BQKc0vtruZQ9gCmVWblpna4o7YAoB3wbhYxmo6yl7mEKbhQ1rGFfYD9zoVR116A+eYpRxXIFAOCAfktf+G+vjfhduR4nJyelXEZv+0rLHGU4RqFPATKRLE4i+MN3jqJPX5Jn+D8D2UddXt+DhMXmRfE3/YDCArDl7T29JcRBS5yF6DecaN9WGvFbLqt3xDIzz5+xTK87NiPJzMw0yGCM08vKYd2pGOeXykYNPWJRVBX/RtSgjVVTwlT40+Qz9qfG8moVKF1Xi0LGnm1Ka4HFFzVAGX2rdfd8KCttloyI5HnVVDrPYzCRS9N9gY/ZHA9qlA+EwPWNp+9l0faloa7FraxT2MxVW6Yoc4PCn17LMiVl59+DzbF44x+x4++ZFUq2MByYYcEWlhUzvNjC8vT09o/JWnRKlrmCyTQldERzBAratLptqiJOztw0tkiXsHwFQpILzRbOPmZN+zZ2NjIbdJFklWqXbb8WOk5SRovJ9aaDKdPzB883gCkTHZi8Mqa3ZNrj46UL/1CkAnq0qDETLImEipTupVgvzI8Rp1CSzCQCcTHqILodW+fVNNgq7F6Iuo5PwSWo9PT0/i0+BD1obQZmPsd/0kn3mwX43Q5speR+rDnjCYl8V8gO9Rt/wMjigqFtMd9OA46VycmaUPK8MOLOxUCfZHneiLXM8+RZLyolT47i5SafGXnMC/AEjvd03r3UQ99EjuBk1o6A+s1Q7dAXMqdNpT2xLpR8azuEm/EozBOIre+NZ/y44zuq4zEoh6gw42fMzFnKfv/iDUvBIQute8Q3ZQgNI/CTj0D/6DHcGr2JSJ9eXCTbezHfB2cQv67U/ctbND+y32cTMWSzNhEfjU2kRmMTOynu+cU++6gofAyULNxGcHE8jF9iP0rIey3exfg19U3JIXPzntG7N6a3Gy3RdiIHbymLui3ut+3kh9FXWFjKKwJe7UOnMIFAa0G0QJlWbn7MUlwP7h6I4e8gnAOfrTeePewreC+tiH2ysDsXcM1cD0+ylBhE3A4lYLmfbNJrtIEwAYBLQPhiFnRqzNoOBkHP09FEIY8SAvW6FeWEw0QZZ1ZBDbXmIW/P4XTb+FXLyTSCqEdBzqIj1K49lnBO4sJyiEuKQspSOv1i++wWTNR/AV8JBmuzmOxnZiWcuSXko0vti1X1IPMO+KBEbgvXCxNyt3uVT7oOcwnzH7h/OiRnyNl2DDZplxszpnPcfM/ePe11VIfhsdRgVrcBEk2Zw5iQcUqEr5ZqB2ZjcPsNzWdzyGClNvNMBYZ7AXgzG4td6cn1pxdDNIXzAFc/j9uhqq0StDYZGnOUEORdSUpZqQIUDGJh0fkU53lX2VQjhwEESGOQRBzHbZbxfO0+mGTlIBPw9wJD8HBGL4Wd0XGUs9G/s+1HDN77tCoc/kquYQOqAjKbtha9Bec5kn4N0asDS9lumCNhvyNRM152MiCxNk7giHU7Bl5r/Lzpg9f3o5jMc1XT/HHQzXjMc9VRHZyJ5PuTyDgQZh4KtucC/SH980Ibj1z90xhd81HXMorEbajjRddFO9uin4xDtqLiXiY/bUAnrEeNtwQbdyuKe75TEndOoISoY8Mvs8ut9Mta3L3zm4uxn+F++Q6miumc4jPd+MQ9QVZH4eO8tTWVnDOqcbCph6Y2PiunfAp4PwPJVy+vYDHTwkwyjwOcuvCdpZ7YSUNKAKW0rPRRNZG1kouA92NsH2VIKw0W7T/YcAMCYxfge+EAf3VMrjVi6+MCg2uYr83Jt6htauWrnHsTWCtC8C3D3InzXz797e5eblJBcAdg5XW1htTfx3U2QM8wvug3uVhqSalZtpKilL1Ktuc6m/OXQkmfEu6fiqWAkLNVMt7PFf+2c+3dX2waseWCdipKS5vMzV7HyVby58uy8SwEiMlqp5Bm5QVEYpUxGzaAyabNe0XxgHZOfX4MBjtJJ74snuVasU2qJgHYoRG2wul45m7jZMYsSUf1S2xxFOveb9v+HtCjYA/fqzKlQ4iUDh98/GeLw5hSnqZLE45ScS+tVL6G1NLWKZVuOyNzrjbHf/PrX0KKbuRolk/7tiNLYkbrfVGTmJyzFbyfGRXF+lfNkmc+iA3z/WBMt8cRp2QKFZDWG33Bo19dXFyQwTJNptRqZIOPRf/YgkXAxQayxIHhRcugaMaE/ZAFFnEsfBEYha0QPz4Uyxu4qLzO4x0199gNEWwXD3bOnhnI3FRa+K7jhDjZykXnjs2As2/oIweZ5ON3kPuoeA3/xF1YLu56IIuPLNJa8nRZD2o1r6OOxwBGZtygJDEsfc8sNZagDRs2dDjwx9SJP9i6PLS3a8nt+Q38xC45UhpYcwbODvywnQJ3D4GG5qNlz/UKzyzbuuRPL/aahB16t91kpmdLVgaHur1T7NzI8jmzdSWjRUbaqCCg+DVVljHbVzHuehLBoiN8iqrGsoQoXJTmRCPJVFdXGr+rOEEhssdSfdXFyQ05PEMDGoP5L3QiqbKFtPEYiO86G4otxBmY9U5eM/phN9esE6BPuRbqXrGb1E7eqp8/+MrHowlzJhOWa7Cb4lunZLAiynXzkU1oMw9I2jjrs5H2DL3YWlw+tfJHo9rt2VOT3Zr/dlXVSN+8DgHC5hMlKjwEnADHbtDKFyhsVYisluL2aDmO+2t/YgOoXrvjmhuYd08fTS32JjFi0aNMGINsdtkqO7hE/p5Zc8IWzsoxG0yBrnXDbV0VgpSFLYwLChBki/E5VAWcQFOKfrI2Qolt4ZaSv8hdTTEHuzuImi1qiOomc54MQDQGFrZ4z/7Csh/ApMX3yeKBlnyfPuCiTY8MRGANcv9hwgKyvPsci7m53aIyt5Vav2X4NTUdTZtDwvcwc5vw7cSg0NoKlKSF04ywNWXW8mItih7HfuOXa9MLdjXk/mcAp9KbOZltHmmYdX1s4XzgZH+sd8ghJ6mWsNdsca2vbaqF9ASiRgiPUu3pHMlOAkrX7PLi4ow6Rl1Vt9oVTAVsDCoDzMCaFnjo8eZHxnabVGbiKV/oVRGEJhEyhl1QgNazyCXR0QrEC/05CFldRy+f/yH6vdSvyfvtZf8KdjrCjcYpGk2FqUFA4mQjEtoeEIHpXMpiU9QyugXxAjGvpHpmUj+MGiOUGgUhzSa6ldG6UkoeGre3ANXgYZpiA5AX9VBKZTLUc7WChZbRW1HhDRkwvlELLnpKYUBVG1DZNfSs68dQxOgjEoCcwLs2yBSIHJKKYMWrOxvAVODRdxkFhsDc04H9Qx8py/OhGV+oHQBB57RvDubKT4k2oCk0UPTdoNwVoJdXhLrRrqgQYJbNVuMoje7xvs1+NJ+bcu751z38j1bAqV0uI/K5biloqmiPJcwTADTlq6IXb7Ggb8C7MafJW65jvPlwRqVF4yvoCoN1zpTnWI2hDzayByGX61uw/mFc7+/FhLcgzL0XvuMqzLZ/8wxAG9GOkaEyNuyGDq0OdcBw94JGvt5vCm/cwqim8f70lO5DMM+XLcZFMDNPOtOC3+kWsaJdAYvc6T1bxGbowJmgUQn9zQ3nps8ef3TzWD0GJWnnKKg+apzLowMljwuEvUFjZcIANpEK02Tlwd6h8nHA/yvfs0JUiE6j2vL3H3xYr0VP525jaBXKKf+03NPNxP76ydjfMQxzMwJZwzrnDQE+LxDYCVFzPqoHn18Ui135r8JwCDwezy5+lYz3zHI81heobmbRbjFb8mLCrXyWBae2GTLgoPV8lHrFUrJMOd5A42xdvfORmitQOX0PuG5p/RwTVwbAi48xTPwhvK1DlwB77q4DZs/7uwEF7xW1AyKqgFge5ZXnedkWeZ4EdJko8bDLkMTM34uEhazaCiuA18xd0mSpzStvqChXoRyN2dsefrhERhcLCLKPoaRbl+wjKEgmwe4aIeCMrmeygyQkwHZ+IDSeFuKTg3TmLiiK+icQY8J3Rs7nzN2L2mZiqjGH2BitZKk5u+QMgxuZa0DJrhkPQbklph8kVyRBpgNkd9iSEQqfX6Llx4Dh8nNOzaM+f45rWlhWFJ3HjIi+4JeB2l9mGfSzo/8i+mrVRt3qv//owHn99K//wM9P//57ERWrn/71t03U/PTvv1bBl38GtyYjvD8CC3Wc3ESpBEX1JojaZ/Mx4YNRUYoaovWgNC2IKCA8kRg7YH2RDDZEmxCdPFQtRBk2y2BTXAzC24YxD62in1hqnow207MxkiPE5tEXjaYY7yrJUZd99YFtA4nQaLr0ZLNDyMibkl9DBAifZDFoNH6UDJL9ySmlB4uQAXZt/QDZc5IGkhXag8nUpwbTqzky9nAQJ6fEITSjpgYgbXG0mulZmgJdOtM9/3nEcV6O3aNwnnIHvDdBcEPWfhQ+U/k8H5eZYUr/GTsLwkiWzOiPrJr+HMdmtjq2vxO8d0ZDoYcbJThwcOPs9/o09E453Ub1HPbEwk6rPMVYTloEskzBpHeRuxVrdJNBpWvkG5S2kt3FHr+uI8M0ssBU3wwubb7fjlsNzfXWhxu6NYvMscYRKMj+8ZMP4z3iSfgQxO1e1SHcPTmp8KqJKS5xzvIcQ4U8h0nhqU10RRngC6CIKYYApv8DUEsDBBQAAAAIABOzdl0pJTRkew0AAAonAAAXAAAAbTFfcmVzdW1lX3JlZmVyZW5jZXMucHmlGl1v47jxPb9C3RfKOVnZvcMdUC9UYHubxR2K9oLNtUDhGgQt0TZrWdSKkhPvYv97Z4akRMtKNm3zEMvUfHG+h/SrV68+StMdZNTIjWxklUsT6SoS0VE2aqNkAS9Mqxt4yHVTd+ZtVEl4B8t5Ix7KSLUmUlXdtSZ99erV1abRh6gQrchLYQwQU4daNy2A16XIZf9etgqYupf+e4L/PuvKgdWi3ZVq7aHu4OuVexbNthaNkclOGIRJ/m10lWhDFCx215TwIiUwTwLWTF2qNqFV/slcWVjRPKojr1UtS1XJdA162B1Es0+brqpgsw79z379XjZH5XczQrYoaV4qj5bvZL6vtapafhSlgt3q5jlU04I+PHKpRcF13SpdmeQe39yjOZ7Df2hUOwh9161LZXbSsTy84WvR5jsuH2sw8UFWrYe87Vc+iByYnJJf2rZ+1xWgsPfK7H9bG9i3bBJiwFHlI5qmq4mS1xcu3jnpPpJwI4SDbBuV914CngjqVZ8lB+dR2wplubq6f/fhlv9y++797cf77AtrAOc0FxvYI0sY+g58WMng4agE/H+c5wLUTk/i8LmaN/JTB46sCpZcRWd/HkI2DYhwqi1SA2RLdVDtnP6P1hp5EKpS1fZi3ciWfb26uiL3jz76oOoVGfdPswUJUshNxDkQazmPjSw3SaEgtlD9DgL/8EXav8gwFOIBbgDrwILxLD0jOODdsB1wnwtkn6L5Sja76qV4aERtJSjFWpYJ2Gani6RtRGXQOoE4REEWmedHqE9g9UhqExHIHzKw1kFUrcq5yXe6FA1bgFG7pvKEexyUKxdlGUPkUmzrrk12UhSyMcm1fAQ2gVj4V+oH2WRf9ik9xLPFMdroJtonR0hTkUNNgcnBxLOvZ6h7ecoIK93KNka/qNUcFsHIbHYGWWNM5dxRA27nXCwRxwP3vcfVmHXgpXOxBacGkiLPZY0Pua5aWJqT741EWuvilNE2l69XSImeI1lCTvsbJsoQGCNGgBolBIk1A1s4czDQH1ugEpkTmi3ONzEOiyE+UAa2wP9pIXNdSLsnXIiUiSrdkiiDUOdbAN/TTTEhEqZ3kolSMlp4luLatCQMilFZoNgudcfnmBDeEAnWcg40Wa5mTxBz6YD3KmMLV0lSsxPf//hTjOGRFt2hNnEPlBhwaA4eYbLfm07OUght0scs3cnHQm2BZPwUSzQ+t8ZfBF4W+MQsYUCb15hFEGqtdRnDylMUEfiAaVQaLqujajQlzAAxElVBbp1pkzoQy/b+e/7u7lf+l9t/sqfIu3DjBgxYoeZ9/DEoUA1EKRfAy9futNIPsa/fadfms1QZDUEBEo4DzeFnhHfQ4EG6Unl8HmOQ5BcXgoFqaiiEMnOJ4rm8MIGMnph2NcocY5XtTOYppvY7R4MmfnGI8EkVDUnGtE18nKWuy0HdJ2z58fb9u59/v32/YjYJyJONEQs8STFIIr1go5xFpDxjBAzL49epXVNm9eTO3stHzEFQ+PFDYd9ncG1K76i6JaMiySlTrTL8iAEc600lDpLzSzyhRiw3qoJ8PmlaywJLVO9zqwsfmTvneRIfGGC7Q965yl7mnhfEzutmetjDcwyZB8LLxn4iHxXkD723meCCwINqd9FF+XVpZw7umUtfgVNdyypmArwE1A/OIcXhUj0kFb2z/V2YoOzWZ9+xf1VBoXKGx/J52ZDYrt/1efFF5xc0J+tOlYXtC3wbOjS0Jqi+B110pTR9U2ARp3B6FHR3aJSpB4UC6SikwyyS2jbRwiQT72t8BpB0A9tci3w/agcsZnoQj+APEEA12O/NFEQhS3HKfkjH6nM8ew3yn7GxLH6lkee3ynsyaqpQJtfQhVplXcOMArnoev+AD7MFhUL0D1F28hajKGb95PWpAxcxw8hFvWvR0zu97aetdanzvSyclZEp7R77aIFT1//K2nH0ZL7N0OiuAf2/lN89gUeFfqhwoPFko6JroIsOhs85hMQJkxWIh82p13rvuKRz56VPddAb78QUf+5b5j7/D8cOqL3Eu7edgMbnwmHiixD1ZcMxSnrDZ0Qi8WZxX53u7TfUEWmgbnQLeQYSnx3H40Zr37NjW2UyGAPbmJZvWM8C8k+zLfU6ZtdsNhugvzsDB18AQAd3c33jJaIc5vEwmvtkRwE9jXxcvp7/cQXsBhWrzYAJiZljup0tSJAUqrmsimHcuZDXafEL1tUakkMpWnWUvNVWBbNxa4cwMFSvTy048Hnvdl6qYD817oPkQBlrlG2jSokdjdM7Re2J2wgKde+9ym2TKsIwwJMM6OxOSTT0p+YTNLQwgTpUnOi5KlxhAKwzjZmeDkc4WwUg8dpOhhpzGjtqAU0mDL59aYRKA1uz6FB7CghBbqcBQWzjURL9dv4QSK2uIaJrd9xQBMceQVECr6tlFkhOC4Hia1Fjw0XLS0YfbLVktMxbYfbQFIRKwF1anBekOYhgChGHYW0yyn2BsHZEg0aVQsEKjUcrHPNTCWb2ABwNaTuBYN6tCvkIyF/iZskg8cmGrRJ4tIc2qrDfTCtr+wSOZED7bAUJlHyvQfN5Fl/PiibJb12Ttn4xx8b0YuDrvg68z129zzBOaBoS4mvqY3vFJDjczc4RwQKxB8BpkCZBENCvLZn1RrbCsb+DzRjDEADt1mcRGklGyg//RvBLC7vqdYZt+6EuJYTO6tvAqOHpTvX5GulYjB2GhixHO9opdJMTu8wkHgKM9pxol5LZdD9lGVcIWOLQQx+6VCRmDFuGAzvZlcFKQ9bwxkpYV7lzPcwc053pt7WHh8YnrzonB/AXFQwcMMXnqo0CPpGNsECNPq80WX+g6atvSicoj21CEXreXTb6AbdjY5fSxzi5gVqolca+BBP3w5JREqfzTVCkD1GrlaFuhoHRW2JCPVg2vMSpKxYoiGWFT3iiMOmT05q1/YQ9dI9gFlLYkkQbAWUJV1u5hfng5LpJm9GZ7xJsY8WHfstWKltmYOxF7WPxTa59E5VR0FvZLGwPZI8hqcYm9rF/ZY0AW8e3WDTN6VCqah/PfHhOlb3ZUF4R7EmQntigsheUKPQ0iE7sN/tO2x61oxHXwkjnbs6tsvDofbJKu+rs9+rdUXct+g9tHTjr8ggb+kN2/v050e/FEUMA4KEr2oCpzNsISvjFnUy0lhv86NvlkSCgqgP02ZSgnDl97D/H/q+IYzVzrj0bp9CfY5yWpL9T1e5gwM49/UGIwRmGTaNE4/5sePsyc9JFwaEzLew+AlUbVcgIhHD3U6S2SSnI+hj+l8T/Dt9FtAFJdp7D0MhiGpEQcxBbR93HE9J/Qe9HcNZM2RMtuuthezkvDxssCB15ZRNXCkHYvdXujiYLL2wCPr4D8tMQxtkThwK0D0YDOEuIeeKpO5npRiebuOWJp+e12HOfJe6+SH2W2ZvXr588fPZ/7t5MuiY3m7pSi13Ny0WNh7XQeH3tZz3Q/Basa6BXMUZsZeBqw2VWoMgb5jHshJPg8TVh4vk1PUBdpIPE/+oYNKhnMPi2XpxkU3ZmF9gaZW5hmIZaH4gK3g5NiMzcPR989AMohOtWZmxI7JCd+meON1Scrqh4u0Mv12WRvXlW69jPc7pk84dx2U9gKK+WzD8MO/J6h6bTyslWmXsaD7zhMk/669ZsfMEagx4E9AEcy1oDls7S1z/OhvtZjodbYg0Fo+dgteYrWpCKMxuXdgGrV2jtniLojaIZEmcB+nwQzcFXBaf8iW3Se2hwMF6XQTOBATGj/IFPwURc6q1hN1bGG2Yx++NAGouwn8FRDi84ENkd/NI1L8y5qo5nlu1wUZpN3Z7GTsj02YEqsSK4+NEdEBtIAGwwG7LVWzoBgvgKQQLHA+KbrixP3MFxO6M4NVmt03WQMKBHRLdDIFuwJ86BEqYbtcWDYz6YHjPBAWwJTenigyiNPHdnZrXLFs4TGKT/lk732cLrpF9K7IwMNAVGew8Qro7IQy+eI3OnwoBmuJ7AlnD4HkOFq2O5h7Dth5oF2eRPr20ld+rPaDFhgxnYYngeUXVZEixoE+YgyvBmaaHAHdinDp7a0xjWLQ+A50zG9Y13YCxRbdFErgY+VQSTS2SL4TFHrHo/wPfuMbF3B/6cF1IdtBtB0V6Usort5bfz9xFRypBgpSX7HVCjwEPxVqTsCmiEsNPw3jic0N64iYYOTdOL3xj0HIay6MdJHAkLLe1xzUHsZXT3/oP1fjdUROIIvT1muWcI33+PHSL+dgc6GauAqKaf6dCNQxL9MLdp3J+303l3Qg6FWzLUchp0rK4EvFxDidAPVcpWtoo+USZthM7tkquV9kvfguHG7NLyOQ9ZTfRldx480LSfurzKcV4AbToqZyeCzm2dQG4Ewl9uxFAkjuFkU2f+F0Xpu2bbocnv8FsDFcjkkG7RThnnhc45d6ekqSig9jvomM3n1HkmeCtGk1HimuciqOsTSF01x1+lvAyYNjIHzX+DD57DZ7X98RNSMLTjs0LmtJJdjoQIn1K1tE82gdrn0YRnu5jgKurLfuFsvV/Z20w7N09ltYmYH9whTNij9Axt1JSZh3QOllZ4J2AvJbOMcY5W55w5H7s/wUx/uAX4mNwB6P0HUEsBAhQDFAAAAAgAE7N2XU7uIpf5DAAAFiYAABMAAAAAAAAAAAAAAIABAAAAAG0xX2JhdGNoX3N1cHBvcnQucHlQSwECFAMUAAAACAATs3ZdxSEFtcwFAADSEgAAEwAAAAAAAAAAAAAAgAEqDQAAbTFfYmF0Y2hfbWV0cmljcy5weVBLAQIUAxQAAAAIABOzdl0WzqY1mgIAACcHAAAaAAAAAAAAAAAAAACAAScTAABtMV9leHBlcmltZW50X3NjaGVkdWxlci5weVBLAQIUAxQAAAAIABOzdl0HGF4OaA8AAIwxAAAWAAAAAAAAAAAAAACAAfkVAABtMV9iYXRjaF9leHBlcmltZW50LnB5UEsBAhQDFAAAAAgAE7N2XSklNGR7DQAACicAABcAAAAAAAAAAAAAAIABlSUAAG0xX3Jlc3VtZV9yZWZlcmVuY2VzLnB5UEsFBgAAAAAFAAUAUwEAAEUzAAAAAA=='''
if not (tools_dir / 'm1_batch_experiment.py').exists() or not (tools_dir / 'm1_batch_support.py').exists():
    print("Đang cập nhật module m1_batch adapter vào tools/...")
    z = zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE_B64)))
    z.extractall(tools_dir)
    print("✓ Đã đồng bộ các công cụ batch benchmark vào tools/!")

# Thiết lập PYTHONPATH
sys.path.insert(0, str(CODE_ROOT / 'src'))
sys.path.insert(0, str(CODE_ROOT / 'tools'))
os.environ['PYTHONPATH'] = f"{CODE_ROOT / 'src'}:{CODE_ROOT / 'tools'}:{os.environ.get('PYTHONPATH', '')}"

from arxiv_pipeline.contracts import SCHEMA_VERSION
print(f"✓ Pipeline Contracts Schema Version: {SCHEMA_VERSION}")


## 3.1. (Tùy chọn) Phục hồi Checkpoint từ Google Drive (Restore Checkpoint)

> **Chỉ chạy cell này khi bạn chạy tiếp phiên 2 hoặc vừa mở lại Colab sau khi ngắt kết nối!**
>
> - Dải 5,000 bài mất ~13.5 - 14.5 tiếng do quy định arXiv API $\ge 3.0$s, trong khi phiên làm việc Colab giới hạn tối đa 12 tiếng. Giảng viên đã cho phép chạy qua 2 phiên (Q&A Hàng 15).
> - **Nếu bạn chạy lần đầu từ bài 1**: BỎ QUA cell này và chạy thẳng Cell 4.
> - **Nếu bạn chạy tiếp phiên 2**: Chạy cell này để giải nén lại dữ liệu phiên trước từ Google Drive vào máy ảo chỉ trong **~15 giây** (thay vì 55 phút nếu copy file lẻ). Sau đó chạy Cell 4, pipeline sẽ tự động đọc `state.sqlite`, bỏ qua các bài đã xong và chỉ crawl các bài còn lại!

In [ ]:
import os, subprocess, shutil
from pathlib import Path

# ==============================================================
# 📥 PHỤC HỒI CHECKPOINT TỪ GOOGLE DRIVE (CHỈ DÙNG KHI CHẠY TIẾP PHIÊN CŨ)
# ==============================================================
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        if not DRIVE_MOUNT.exists() or not os.path.ismount(str(DRIVE_MOUNT)):
            print("Đang kết nối Google Drive...")
            drive.mount(str(DRIVE_MOUNT))

        candidates = [
            DRIVE_OUTPUT_DIR / f"arXiv-checkpoint-{OWNER}.tar",
            DRIVE_CHECKPOINT_DIR / f"arXiv-checkpoint-{OWNER}.tar",
            DRIVE_MOUNT / "MyDrive" / f"arXiv-checkpoint-{OWNER}.tar",
            DRIVE_MOUNT / "MyDrive" / "arXiv-data" / f"arXiv-checkpoint-{OWNER}.tar",
        ]

        found_tar = None
        for cand in candidates:
            if cand.exists():
                found_tar = cand
                break

        if found_tar:
            tar_size_mb = found_tar.stat().st_size / (1024**2)
            print(f"📦 Tìm thấy file checkpoint: {found_tar} ({tar_size_mb:.1f} MiB)")
            print("Đang giải nén checkpoint vào máy ảo (chỉ mất ~15-20 giây)...)")
            WORK_ROOT.mkdir(parents=True, exist_ok=True)
            res = subprocess.run(['tar', '-xf', str(found_tar), '-C', str(WORK_ROOT)], capture_output=True, text=True)
            if res.returncode == 0:
                sqlite_file = CORPUS_DIR / 'state.sqlite'
                if sqlite_file.exists():
                    print(f"✓ Đã phục hồi SQLite checkpoint: {sqlite_file} ({sqlite_file.stat().st_size / 1024:.1f} KiB)!")
                print(f"✓ Phục hồi thành công! Thư mục {CORPUS_DIR} đã sẵn sàng.")
                print("👉 Bạn hãy chạy tiếp Cell 4: Pipeline sẽ tự động bỏ qua các bài đã xong và tiếp tục crawl các bài còn lại.")
            else:
                print(f"⚠️ Lỗi giải nén: {res.stderr}")
        else:
            print(f"ℹ️ Không tìm thấy file checkpoint trên Google Drive.")
            print("   (Nếu đây là phiên đầu tiên, điều này hoàn toàn bình thường. Bạn hãy bỏ qua và chạy thẳng Cell 4)")
    except Exception as e:
        print(f"⚠️ Lỗi kiểm tra checkpoint: {e}")


## 4. Thực thi Benchmark M1 Toàn trình (End-to-End Crawl & Profiling)

Chạy toàn bộ pipeline M1 cho dải bài báo của `OWNER` với bộ đo đa chiều:
* **Bộ đo RAM**: Lấy mẫu mỗi 50ms qua `psutil process_tree_rss`.
* **Bộ đo Ổ cứng**: Quan sát dung lượng tại các mốc tải và giải nén source.
* **Bộ ghi vết HTTP**: Ghi lại chi tiết từng request vào `http-audit.jsonl`.
* **Đo đạc thời gian**: Ghi nhận `wall_seconds`, `discovery_seconds`, và độ trễ từng bài.
* **Cơ chế Checkpoint tự động**: Nếu thư mục đã có `state.sqlite`, pipeline sẽ tự động ghi nhận bài đã làm thành `dùng lại (success)` và chỉ thực thi các bài chưa hoàn thành.

In [ ]:
import time, shutil, subprocess, sys
from pathlib import Path

# Giữ nguyên dữ liệu cũ nếu đã có checkpoint, chỉ tạo thư mục nếu chưa tồn tại
WORK_ROOT.mkdir(parents=True, exist_ok=True)
CORPUS_DIR.mkdir(parents=True, exist_ok=True)

BENCHMARK_SCRIPT = CODE_ROOT / 'tools/m1_batch_experiment.py'

print(f"🚀 BẮT ĐẦU CHẠY BENCHMARK M1")
print(f"   - Owner: {OWNER}")
print(f"   - Số lượng: {COUNT} bài báo")
print(f"   - Chế độ: {VARIANT.upper()} (S2 Batch Size: {BATCH_SIZE})")
print(f"   - Thư mục output: {CORPUS_DIR}")
if (CORPUS_DIR / 'state.sqlite').exists():
    print(f"   - Chế độ Checkpoint: RESUME TỰ ĐỘNG TỪ PHIÊN TRƯỚC (state.sqlite sẵn sàng)")
print("=" * 75)

cmd = [
    sys.executable, str(BENCHMARK_SCRIPT),
    '--variant', VARIANT,
    '--owner', OWNER,
    '--count', str(COUNT),
    '--batch-size', str(BATCH_SIZE),
    '--run-id', RUN_ID,
    '--root', str(CORPUS_DIR),
    '--stop-after-seconds', '86400'
]

start_clock = time.monotonic()
proc = subprocess.Popen(cmd, cwd=CODE_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

for line in proc.stdout:
    l = line.strip()
    if l:
        print(l)

proc.wait()
elapsed = time.monotonic() - start_clock
print("=" * 75)
print(f"✓ Quá trình thu thập hoàn tất trong {elapsed:.1f}s (~{elapsed/60:.2f} phút)!")
print(f"✓ Mã thoát (Exit code): {proc.returncode}")


## 4.1. (Tùy chọn) Lưu Checkpoint sang Google Drive (Save Checkpoint)

> **Chạy cell này khi bạn muốn tạm dừng hoặc trước khi tắt máy Colab (khi sắp hết 12h)!**
>
> Thay vì copy từng file lẻ mất gần 1 tiếng qua mạng Google Drive FUSE, cell này sử dụng lệnh `tar` đóng gói toàn bộ thư mục `corpus` thành một file duy nhất và lưu thẳng lên Google Drive chỉ trong **~15-20 giây**.
> Lần sau mở Colab mới, bạn chỉ cần chạy cell `3.1 Phục hồi Checkpoint` là tiếp tục ngay lập tức!

In [ ]:
import os, subprocess, shutil
from pathlib import Path

# ==============================================================
# 💾 LƯU CHECKPOINT TIẾN ĐỘ SANG GOOGLE DRIVE (KHI CẦN TẠM DỪNG / TRƯỚC HẾT 12H)
# ==============================================================
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        if not DRIVE_MOUNT.exists() or not os.path.ismount(str(DRIVE_MOUNT)):
            print("Đang kết nối Google Drive...")
            drive.mount(str(DRIVE_MOUNT))

        DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
        CHECKPOINT_TAR_SAVE = DRIVE_OUTPUT_DIR / f"arXiv-checkpoint-{OWNER}.tar"
        CHECKPOINT_TAR_PARENT = DRIVE_CHECKPOINT_DIR / f"arXiv-checkpoint-{OWNER}.tar"

        if CORPUS_DIR.exists():
            print(f"📦 Đang đóng gói checkpoint từ {CORPUS_DIR} sang {CHECKPOINT_TAR_SAVE.name} (chỉ mất ~15-20s)...)")
            res = subprocess.run(['tar', '-cf', str(CHECKPOINT_TAR_SAVE), '-C', str(WORK_ROOT), 'corpus'], capture_output=True, text=True)
            if res.returncode == 0:
                try:
                    shutil.copy2(CHECKPOINT_TAR_SAVE, CHECKPOINT_TAR_PARENT)
                except Exception:
                    pass
                tar_size_mb = CHECKPOINT_TAR_SAVE.stat().st_size / (1024**2)
                print(f"🎉 ĐÃ LƯU CHECKPOINT LÊN GOOGLE DRIVE THÀNH CÔNG ({tar_size_mb:.1f} MiB)!")
                print(f"  Vị trí lưu: {CHECKPOINT_TAR_SAVE}")
                print("  Lần sau mở phiên mới, bạn chỉ cần chạy cell '3.1 Phục hồi Checkpoint' là pipeline tiếp tục chạy mượt mà!")
            else:
                print(f"⚠️ Lỗi khi tạo file tar: {res.stderr}")
        else:
            print(f"⚠️ Thư mục {CORPUS_DIR} chưa tồn tại!")
    except Exception as e:
        print(f"⚠️ Lỗi sao lưu checkpoint: {e}")


## 5. Bảng Tổng hợp Metrics Chuẩn Rubric Đề bài (`context/crawling_new.md`)

Trích xuất trực tiếp từ file `experiment-metrics.json` và định dạng thành bảng chuẩn khoa học cho báo cáo (hoàn toàn an toàn với giá trị null).

In [ ]:
import json
import pandas as pd
from IPython.display import display, Markdown

def safe_fmt(v, fmt='.1f', scale=1.0, suffix='', na='N/A'):
    if v is None: return na
    try: return f"{v * scale:{fmt}}{suffix}"
    except Exception: return str(v)

metrics_file = CORPUS_DIR / 'experiment-metrics.json'
assert metrics_file.exists(), f"Không tìm thấy {metrics_file}!"

metrics = json.loads(metrics_file.read_text(encoding='utf-8'))
bench = metrics.get('benchmark', {})
timing = metrics.get('timing', {})
assign = metrics.get('assignment', {})
refs = assign.get('references', {})
sources = assign.get('source_sizes', {})
versions = assign.get('versions', {})

# 1. Running Time
time_df = pd.DataFrame([
    ["Tổng thời gian toàn trình (End-to-End Wall Time)", f"{safe_fmt(bench.get('wall_seconds'), '.2f', suffix='s')} (~{safe_fmt(bench.get('wall_seconds'), '.2f', scale=1/60, suffix='m')})", "Bao gồm discovery, tải sources, references, trích xuất và đo đạc"],
    ["Thời gian khám phá danh mục (Discovery Time)", safe_fmt(timing.get('discovery_seconds'), '.3f', suffix='s'), "Truy vấn arXiv Atom API để xác định danh mục bài"],
    ["Thời gian trung bình xử lý 1 bài (Mean Latency)", safe_fmt(timing.get('mean_paper_seconds'), '.2f', suffix='s/bài'), "Trung bình thời gian metadata + source + refs"],
    ["Độ trễ bài nhanh nhất (Min Paper Latency)", safe_fmt(timing.get('min_paper_seconds'), '.1f', suffix='s'), "Bài có ít phiên bản / dung lượng nhẹ"],
    ["Độ trễ bài lâu nhất (Max Paper Latency)", safe_fmt(timing.get('max_paper_seconds'), '.1f', suffix='s'), "Bài có nhiều phiên bản hoặc source lớn"]
], columns=["Chỉ số Thời gian", "Giá trị đo đạc", "Mô tả"]).set_index("Chỉ số Thời gian")

# 2. Memory Footprint
mem_df = pd.DataFrame([
    ["RAM cực đại (Peak RAM / Maximum RSS)", safe_fmt(bench.get('peak_ram_bytes'), '.2f', scale=1/(1024**2), suffix=' MiB'), "Đo đạc qua psutil process_tree_rss (chu kỳ 50ms)"],
    ["RAM trung bình (Mean RAM Consumption)", safe_fmt(bench.get('mean_ram_bytes'), '.2f', scale=1/(1024**2), suffix=' MiB'), "Mức tiêu thụ bộ nhớ ổn định trung bình"],
    ["Ổ cứng cực đại (Peak Disk Storage)", safe_fmt(bench.get('peak_disk_bytes'), '.2f', scale=1/(1024**2), suffix=' MiB'), "Ghi nhận tại các mốc tải & giải nén tar/gz"],
    ["Dung lượng Corpus cuối (Final Output Bytes)", safe_fmt(bench.get('final_output_bytes'), '.2f', scale=1/(1024**2), suffix=' MiB'), "Dung lượng dữ liệu lưu trữ sau khi lọc sạch"]
], columns=["Chỉ số Bộ nhớ & Ổ cứng", "Giá trị đo đạc", "Mô tả"]).set_index("Chỉ số Bộ nhớ & Ổ cứng")

# 3. Scraped Data Statistics
before_kb = safe_fmt(sources.get('mean_before_filter_bytes'), '.1f', scale=1/1024, suffix=' KiB')
after_kb = safe_fmt(sources.get('mean_retained_bytes'), '.1f', scale=1/1024, suffix=' KiB')
b_val = sources.get('mean_before_filter_bytes')
a_val = sources.get('mean_retained_bytes')
reduc_str = f"{(1 - (a_val / max(1, b_val)))*100:.1f}%" if b_val and a_val else "N/A"

stat_df = pd.DataFrame([
    ["Tổng số bài yêu cầu (Requested Publications)", f"{assign.get('publications', 0)} bài", "Mẫu được chỉ định theo rubric"],
    ["Số bài thu thập hoàn tất (Completed Papers)", f"{assign.get('papers_complete', 0)} bài", "Hoàn thành 100% các version & metadata"],
    ["Tỷ lệ thành công (Paper Success Rate)", safe_fmt(assign.get('paper_success_rate'), '.1f', scale=100, suffix='%'), "Tỷ lệ bài đạt chuẩn hoàn chỉnh"],
    ["Độ bao phủ phiên bản (Versions Coverage)", f"{versions.get('covered', 0)} / {versions.get('expected', 0)} phiên bản", f"Unsupported: {versions.get('unsupported', 0)}, Unresolved: {versions.get('unresolved', 0)}"],
    ["Dung lượng TeX Trước / Sau lọc", f"{before_kb} -> {after_kb}", f"Giảm {reduc_str} dung lượng nhờ lọc ảnh"],
    ["Số bài trích xuất References thành công", f"{refs.get('fully_fetched_papers', 0)} / {assign.get('publications', 0)} bài", f"Tỷ lệ query S2: {safe_fmt(refs.get('query_success_rate'), '.1f', scale=100, suffix='%')}"],
    ["Tổng số trích dẫn thu thập (Raw References)", f"{safe_fmt(refs.get('total_raw_references_complete_lists'), ',d', suffix=' refs')}", f"Trung bình: {safe_fmt(refs.get('mean_all_references_per_fully_fetched_paper'), '.1f', suffix=' refs/bài')}"],
    ["Trích dẫn arXiv (arXiv References)", f"{safe_fmt(refs.get('metadata_complete_arxiv_entries'), ',d', suffix=' bài')}", f"Trung bình: {safe_fmt(refs.get('mean_arxiv_references_per_fully_fetched_paper'), '.1f', suffix=' arXiv refs/bài')}"],
    ["Tỷ lệ chuẩn hóa Metadata trích dẫn (Macro / Micro)", f"{safe_fmt(refs.get('metadata_macro_success_rate'), '.1f', scale=100, suffix='%')} / {safe_fmt(refs.get('metadata_micro_success_rate'), '.1f', scale=100, suffix='%')}", "Đầy đủ title, authors, date, s2_id"]
], columns=["Chỉ số Thu thập Dữ liệu", "Giá trị đo đạc", "Mô tả"]).set_index("Chỉ số Thu thập Dữ liệu")

display(Markdown(f"### 📊 BẢNG TỔNG HỢP CHỈ SỐ BENCHMARK — OWNER: `{OWNER.upper()}` ({VARIANT.upper()})"))
display(Markdown("#### ⏱️ 1. Running Time"))
display(time_df)
display(Markdown("#### 💾 2. Memory Footprint"))
display(mem_df)
display(Markdown("#### 📈 3. Data & Crawling Statistics"))
display(stat_df)
display(Markdown(f"**HTTP Requests Audit**: Tổng số {metrics.get('http_attempts_actual', 0)} requests. Phân phối: `{metrics.get('http_counts')}`"))


## 6. Trực quan hóa Đồ thị Khoa học (Jupyter Book Figures)

Tạo 4 biểu đồ trực quan sắc nét để đưa vào báo cáo Jupyter Book:
1. **Phân phối độ trễ xử lý từng bài (Paper Latency Distribution)**.
2. **So sánh dung lượng Source TeX trước và sau khi lọc bỏ hình ảnh**.
3. **Cơ cấu trích dẫn (Trích dẫn arXiv có đủ metadata vs Trích dẫn ngoài)**.
4. **Nhật ký HTTP requests phân loại theo dịch vụ và mã trạng thái**.

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(f'MILESTONE 1 PIPELINE BENCHMARK ANALYSIS — {OWNER.upper()} ({VARIANT.upper()} MODE)', fontsize=16, fontweight='bold', y=0.98)

# 1. Latency Distribution
latencies = [v for v in timing.get('per_paper_seconds', {}).values() if v is not None]
ax1 = axes[0, 0]
if latencies:
    ax1.hist(latencies, bins=min(20, max(5, len(latencies))), color='#3498db', edgecolor='black', alpha=0.8)
    ax1.axvline(np.mean(latencies), color='red', linestyle='--', linewidth=2, label=f"Trung bình: {np.mean(latencies):.1f}s")
    ax1.axvline(np.median(latencies), color='green', linestyle=':', linewidth=2, label=f"Trung vị: {np.median(latencies):.1f}s")
    ax1.set_title('Phân phối Thời gian Xử lý Từng Bài (Paper Latency)', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Thời gian (giây)')
    ax1.set_ylabel('Số lượng bài báo')
    ax1.legend()
else:
    ax1.text(0.5, 0.5, 'Chưa có dữ liệu latency', ha='center')

# 2. Source Size Reduction
ax2 = axes[0, 1]
before_kb = (sources.get('mean_before_filter_bytes') or 0) / 1024
after_kb = (sources.get('mean_retained_bytes') or 0) / 1024
reduc = (1 - (after_kb / max(1, before_kb))) * 100 if before_kb else 0
bars = ax2.bar(['Trước khi lọc\n(Raw Source)', 'Sau khi lọc\n(TeX/Bib Retained)'], [before_kb, after_kb], color=['#e74c3c', '#2ecc71'], width=0.45, edgecolor='black')
for b in bars:
    y = b.get_height()
    ax2.text(b.get_x() + b.get_width()/2.0, y + (max(1, before_kb)*0.02), f"{y:.1f} KiB", ha='center', va='bottom', fontweight='bold')
ax2.set_title(f'Dung lượng Trung bình Mỗi Bài (Giảm {reduc:.1f}%)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Dung lượng trung bình (KiB)')
ax2.set_ylim(0, max(10, before_kb * 1.2))

# 3. References Breakdown
ax3 = axes[1, 0]
total_raw = refs.get('total_raw_references_complete_lists') or 0
arxiv_entries = refs.get('metadata_complete_arxiv_entries') or 0
non_arxiv = max(0, total_raw - arxiv_entries)
if total_raw > 0:
    ax3.pie([arxiv_entries, non_arxiv], labels=['Trích dẫn arXiv\n(Đủ metadata)', 'Trích dẫn Non-arXiv\n(Khác)'],
            autopct='%1.1f%%', colors=['#9b59b6', '#95a5a6'], startangle=140, explode=(0.05, 0), shadow=True)
    ax3.set_title(f'Cơ cấu Trích dẫn ({total_raw:,} tổng trích dẫn)', fontsize=12, fontweight='bold')
else:
    ax3.text(0.5, 0.5, 'Chưa có dữ liệu trích dẫn', ha='center')

# 4. HTTP Audit
ax4 = axes[1, 1]
http_c = metrics.get('http_counts', {})
if http_c:
    labels = [k.replace('|', ' ') for k in http_c.keys()]
    vals = list(http_c.values())
    y_pos = np.arange(len(labels))
    bars4 = ax4.barh(y_pos, vals, color='#f39c12', edgecolor='black')
    ax4.set_yticks(y_pos)
    ax4.set_yticklabels(labels, fontsize=9)
    ax4.invert_yaxis()
    for bar in bars4:
        xval = bar.get_width()
        ax4.text(xval + max(vals)*0.01, bar.get_y() + bar.get_height()/2.0, f'{int(xval)}', ha='left', va='center', fontweight='bold')
    ax4.set_title(f'Phân phối HTTP Requests ({metrics.get("http_attempts_actual", 0)} requests)', fontsize=12, fontweight='bold')
    ax4.set_xlabel('Số lượng yêu cầu (Requests)')
else:
    ax4.text(0.5, 0.5, 'Chưa có dữ liệu HTTP', ha='center')

plt.tight_layout()
fig_out = CORPUS_DIR / 'benchmark_figures.png'
plt.savefig(fig_out, dpi=300, bbox_inches='tight')
print(f"✓ Đã xuất biểu đồ khoa học: {fig_out}")
plt.show()


## 7. Đóng gói & Tự động Lưu trữ lên Google Drive

Tự động nén toàn bộ corpus kết quả, SQLite, logs và báo cáo thành file zip, tính mã băm SHA-256 xác thực và sao lưu toàn bộ vào Google Drive của nhóm.

In [ ]:
import zipfile, hashlib, shutil
from pathlib import Path

archive_path = WORK_ROOT / f"{RUN_ID}-result.zip"
print(f"Đang đóng gói kết quả vào {archive_path.name}...")

with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in CORPUS_DIR.rglob('*'):
        if f.is_file():
            z.write(f, arcname=Path('corpus') / f.relative_to(CORPUS_DIR))

with zipfile.ZipFile(archive_path, 'r') as z:
    assert z.testzip() is None, "Lỗi tính toàn vẹn file zip!"
    
sha256 = hashlib.sha256(archive_path.read_bytes()).hexdigest()
(WORK_ROOT / f"{archive_path.name}.sha256").write_text(sha256 + '\n')
zip_size_mb = archive_path.stat().st_size / (1024**2)
print(f"✓ Đã tạo file kết quả: {archive_path} ({zip_size_mb:.2f} MiB)")
print(f"✓ SHA-256: {sha256}")

if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        if not DRIVE_MOUNT.exists() or not os.path.ismount(str(DRIVE_MOUNT)):
            print("Đang kết nối Google Drive...")
            drive.mount(str(DRIVE_MOUNT))
        
        DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
        print(f"Đang sao lưu lên Google Drive: {DRIVE_OUTPUT_DIR}...")
        
        shutil.copy2(archive_path, DRIVE_OUTPUT_DIR / archive_path.name)
        shutil.copy2(WORK_ROOT / f"{archive_path.name}.sha256", DRIVE_OUTPUT_DIR / f"{archive_path.name}.sha256")
        shutil.copy2(CORPUS_DIR / 'experiment-metrics.json', DRIVE_OUTPUT_DIR / 'experiment-metrics.json')
        
        fig_file = CORPUS_DIR / 'benchmark_figures.png'
        if fig_file.exists():
            shutil.copy2(fig_file, DRIVE_OUTPUT_DIR / 'benchmark_figures.png')
            
        print("\n🎉 ĐỒNG BỘ TOÀN BỘ KẾT QUẢ VÀ BIỂU ĐỒ LÊN GOOGLE DRIVE THÀNH CÔNG!")
        print(f"Thư mục Google Drive: {DRIVE_OUTPUT_DIR}")
    except Exception as e:
        print(f"⚠️ Không thể sao lưu lên Drive: {e}")
        print(f"File kết quả cục bộ: {archive_path}")
